# Data Preprocessing: Download, Import and Initial Data Overview

This notebook performs the entire data preprocessing pipeline for the Berlin Green Roofs project:
1. **Data Loading**: Import geographic data (shapefiles) for green roofs, districts, roof slopes, and solar potential
2. **Data Validation**: Clean and validate geometries
3. **Building Analysis**: Analyze which buildings have suitable flat roofs for green roof installation based on slope thresholds
4. **Feature Engineering**: Calculate proportions and aggregate roof characteristics by building
5. **Data Export**: Export processed data as QGIS-compatible formats (Shapefile and GeoPackage)

## Load Required Packages

In [ ]:
# Import all needed packages
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import sklearn
import os
from dotenv import load_dotenv
from pathlib import Path

In [ ]:
# Check working directory
cwd = Path.cwd()
print(f"Working Directory: {cwd}")

# Try to find .env in current or parent directories
env_path = None
for possible_path in [Path(".env"), Path("../.env"), cwd / ".env", cwd.parent / ".env"]:
    if possible_path.exists():
        env_path = possible_path
        print(f"✓ .env found at: {env_path}")
        break

if env_path:
    load_dotenv(dotenv_path=env_path)
else:
    print("✗ No .env file found!")

print("Data_path:", os.getenv("Data_path"))

data_base_path = os.getenv("Data_path")

Working Directory: c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\notebooks
✓ .env gefunden at: ..\.env
Data_path: C:\\Users\\User\\Documents\\Uni\\Master_Bauing\\WiSe25_26\\AI_in_Human_Water\\berlin-green-roofs\\data


## Load Geographic Data Files

Load the main datasets:
- **Green Roofs 2020**: Existing green roofs in Berlin
- **Districts**: Administrative district boundaries
- **Solar Potential**: Building solar potential data
- **Roof Slopes**: Roof segment data with slope information

In [ ]:
# Load shape file
shape_file_path = Path(data_base_path) / "green roofs 2020.shp"
green_roofs = gpd.read_file(shape_file_path)

print(green_roofs.columns)

Index(['gml_id', 'importid', 'geb_nutz', 'gruendach', 'ex_int', 'gruen20_m2',
       'gint20_m2', 'gex20_m2', 'gruen20_p', 'gint20_p', 'gex20_p', 'geb_area',
       'nutz', 'ext', 'egeb_nutz', 'egruendach', 'eex_int', 'geometry'],
      dtype='str')


In [ ]:
# Load districts data
districts_shapes_file_path = Path(data_base_path) / "Bezirke.shp"
districts_data = gpd.read_file(districts_shapes_file_path)

# Convert districts data to a pandas dataframe to get an attribute table
# districts_attribute_table = pd.DataFrame(districts_data.drop(columns="geometry"))
print(districts_data.columns)

# Keep only "geometry" and "name" columns in district data
districts_data = districts_data[["geometry","namgem"]]
print(districts_data.columns)

print(districts_data.head())

Index(['gml_id', 'name', 'gem', 'namgem', 'namlan', 'lan', 'geometry'], dtype='str')
Index(['geometry', 'namgem'], dtype='str')
                                            geometry  \
0  POLYGON ((390754.654 5825381.256, 390756.332 5...   
1  POLYGON ((396077.557 5817819.888, 396066.256 5...   
2  MULTIPOLYGON (((399003.49 5834202.526, 399004....   
3  POLYGON ((387115.47 5816898.439, 387112.266 58...   
4  POLYGON ((377248.57 5818041.669, 377119.245 58...   

                       namgem  
0                       Mitte  
1    Friedrichshain-Kreuzberg  
2                      Pankow  
3  Charlottenburg-Wilmersdorf  
4                     Spandau  


### All shape files are exported with QGis. Original Data source:
Solarpotential: https://gdi.berlin.de/geonetwork/srv/ger/catalog.search#/metadata/d721672d-e309-4fcf-8159-92e405637d1a
green roofs 2020: https://gdi.berlin.de/geonetwork/srv/ger/catalog.search#/metadata/08828320-6046-3c60-a290-5be6612356c7

In [ ]:
# Load the XML (.application file)
xml_file_path = Path(data_base_path) / "Solarpotential.application"
with open(xml_file_path, "r") as file:
    xml_content = file.read()
print("XML Content:")
print(xml_content)

# Load the two layers from the application file as shape file
solar_potential_layer_path = Path(data_base_path) / "Solarpotential.shp"
solar_potential_layer = gpd.read_file(solar_potential_layer_path)

print(solar_potential_layer.columns)

# Load the roof slope layer from the application file as shape file
roof_slope_layer_path = Path(data_base_path) / "Dachneigung.shp"
roof_slope_layer = gpd.read_file(roof_slope_layer_path)
print(roof_slope_layer.columns)
print(green_roofs.columns)

# Print number of rows and columns of the shape files
print(f"Green Roofs: {green_roofs.shape[0]} rows, {green_roofs.shape[1]} columns")
print(f"Districts: {districts_data.shape[0]} rows, {districts_data.shape[1]} columns")
print(f"Solar Potential: {solar_potential_layer.shape[0]} rows, {solar_potential_layer.shape[1]} columns")
print(f"Roof Slope: {roof_slope_layer.shape[0]} rows, {roof_slope_layer.shape[1]} columns")

XML Content:
<?xml version="1.0" encoding="UTF-8"?><wfs:WFS_Capabilities version="2.0.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.opengis.net/wfs/2.0" xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:fes="http://www.opengis.net/fes/2.0" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xs="http://www.w3.org/2001/XMLSchema" xsi:schemaLocation="http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/wfs/2.0/wfs.xsd http://inspire.ec.europa.eu/schemas/inspire_dls/1.0 https://inspire.ec.europa.eu/schemas/inspire_dls/1.0/inspire_dls.xsd" xmlns:xml="http://www.w3.org/XML/1998/namespace" xmlns:inspire_dls="http://inspire.ec.europa.eu/schemas/inspire_dls/1.0" xmlns:inspire_common="http://inspire.ec.europa.eu/schemas/common/1.0" xmlns:ua_solarpotenzial_solarrechner="ua_solarpotenzial_solarrechner" updateSequence="512744"><ows:ServiceIdentification><ows:Title>Solarpotenzial - Sol

In [ ]:
# Keep only the most important columns to simplify the shape files
green_roofs = green_roofs[["geometry", 'gruendach']]
print(green_roofs.columns)
solar_potential_layer = solar_potential_layer[["geometry", 'anzahl_obe', 'ist_hochha', 'ist_denkma', 'verschattu', 'verschat_1', 'verschat_2', 'gebaeudefu',  'bauweise_s', 'id',]]
print(solar_potential_layer.columns)
dachneigung_layer = roof_slope_layer[["geometry", 'uid_gebaeu', 'ausrichtun', 'neigung', 'flaeche', 'id0',]]
print(dachneigung_layer.columns)

Index(['geometry', 'gruendach'], dtype='str')
Index(['geometry', 'anzahl_obe', 'ist_hochha', 'ist_denkma', 'verschattu',
       'verschat_1', 'verschat_2', 'gebaeudefu', 'bauweise_s', 'id'],
      dtype='str')
Index(['geometry', 'uid_gebaeu', 'ausrichtun', 'neigung', 'flaeche', 'id0'], dtype='str')


In [ ]:
# Fix invalid geometries before clipping
print("Validating geometries...")

green_roofs = green_roofs[green_roofs.geometry.is_valid]
dachneigung_layer = dachneigung_layer[dachneigung_layer.geometry.is_valid]
solar_potential_layer = solar_potential_layer[solar_potential_layer.geometry.is_valid]
#neukoelln = neukoelln[neukoelln.geometry.is_valid]

print(f"✓ After validation:")
print(f"  Green Roofs: {len(green_roofs)}")
print(f"  Roof Slope: {len(dachneigung_layer)}")
print(f"  Solar Potential: {len(solar_potential_layer)}")
#print(f"  Neukölln: {len(neukoelln)}")

# If problems remain: clip with error tolerance
# green_roofs_neukoelln = gpd.clip(green_roofs, neukoelln, keep_geom_type=False)
# dachneigung_neukoelln = gpd.clip(dachneigung_layer, neukoelln, keep_geom_type=False)
# solar_potential_neukoelln = gpd.clip(solar_potential_layer, neukoelln, keep_geom_type=False)

Validiere Geometrien...
✓ Nach Validierung:
  Green Roofs: 629488
  Dachneigung: 1481759
  Solar Potential: 529946


## Data Validation and Cleaning

Validate and clean geometric data:
- Check for invalid geometries
- Remove invalid features that cannot be processed
- Prepare data for spatial operations

## Building-wise Green Roof Analysis

Since roofs are divided into different slopes, they must first be assigned to each building.
Subsequently, it is determined whether a building has a flat roof section suitable for green roofs.

In [ ]:
# ============================================================
# NEW APPROACH: Group by uid_gebaeu, use existing area
# ============================================================

print("="*70)
print("Calculate proportion of suitable area per building (uid_gebaeu)...")
print("="*70)

SLOPE_THRESHOLD = 15.0

# Define "suitable"
dachneigung_layer['is_suitable'] = dachneigung_layer['neigung'] <= SLOPE_THRESHOLD

# Group by uid_gebaeu and aggregate
building_results_v2 = dachneigung_layer.groupby('uid_gebaeu', as_index=False).agg({
    'flaeche': 'sum',  # Total roof area of building
    'neigung': 'mean',  # Average slope
    'ausrichtun': lambda x: x.mode()[0] if len(x.mode()) > 0 else None  # Most common orientation
}).rename(columns={
    'flaeche': 'total_roof_area',
    'neigung': 'avg_slope',
    'ausrichtun': 'dominant_orientation'
})

# ============================================================
# HANDLING MISSING VALUES (as in DecisionTree_RandomForest_pred.ipynb)
# ============================================================
print("\nHandling missing values...")

# For numeric features: use median (robust against outliers)
numeric_cols = building_results_v2.select_dtypes(include=['number']).columns
for col in numeric_cols:
    missing_count = building_results_v2[col].isna().sum()
    if missing_count > 0:
        median_val = building_results_v2[col].median()
        building_results_v2[col].fillna(median_val, inplace=True)
        print(f"  ✓ Column '{col}': {missing_count} missing values filled with median ({median_val:.2f})")

# For categorical features: use most frequent value
if building_results_v2['dominant_orientation'].isna().sum() > 0:
    mode_val = building_results_v2['dominant_orientation'].mode()[0]
    building_results_v2['dominant_orientation'].fillna(mode_val, inplace=True)
    print(f"  ✓ Column 'dominant_orientation': missing values filled with mode")

print(f"\n✓ Missing value handling completed")

# Calculate suitable_area (only segments with slope <= 15°)
suitable_area_per_building = dachneigung_layer[dachneigung_layer['is_suitable']].groupby('uid_gebaeu')['flaeche'].sum()
building_results_v2['suitable_roof_area'] = building_results_v2['uid_gebaeu'].map(suitable_area_per_building).fillna(0)

# Calculate proportion
building_results_v2['suitable_share_pct'] = np.where(
    building_results_v2['total_roof_area'] > 0,
    (building_results_v2['suitable_roof_area'] / building_results_v2['total_roof_area']) * 100,
    0.0
)

# ============================================================
# VALIDATION AND QUALITY CONTROL
# ============================================================
print("\nQuality control:")
print(f"  Missing values after handling:")
for col in building_results_v2.columns:
    missing = building_results_v2[col].isna().sum()
    if missing > 0:
        print(f"    ⚠ '{col}': {missing} missing values")
if building_results_v2.isna().sum().sum() == 0:
    print(f"    ✓ No missing values anymore!")

print(f"\n✓ {len(building_results_v2)} buildings processed")
print(f"\nStatistics:")
print(f"  Average proportion: {building_results_v2['suitable_share_pct'].mean():.1f}%")
print(f"  Buildings with >75% suitable area: {(building_results_v2['suitable_share_pct'] > 75).sum()}")
print(f"\nPreview (Top 10):")
print(building_results_v2.nlargest(10, 'suitable_share_pct')[['uid_gebaeu', 'total_roof_area', 'suitable_roof_area', 'suitable_share_pct']].to_string(index=False))

Berechne Anteil geeigneter Fläche pro Gebäude (uid_gebaeu)...

Behandlung von Fehlwerten...
  ✓ Spalte 'avg_slope': 16980 fehlende Werte mit Median (23.55) gefüllt
  ✓ Spalte 'dominant_orientation': 16990 fehlende Werte mit Median (-38.80) gefüllt
  ✓ Spalte 'dominant_orientation': fehlende Werte mit Modus gefüllt

✓ Fehlwertbehandlung abgeschlossen


C:\Users\User\AppData\Local\Temp\ipykernel_21168\3676990569.py:36: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  building_results_v2[col].fillna(median_val, inplace=True)
C:\Users\User\AppData\Local\Temp\ipykernel_21168\3676990569.py:36: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained a


Qualitätskontrolle:
  Fehlende Werte nach Behandlung:
    ⚠ 'avg_slope': 16980 fehlende Werte
    ⚠ 'dominant_orientation': 16990 fehlende Werte

✓ 529937 Gebäude verarbeitet

Statistik:
  Durchschnittlicher Anteil: 43.2%
  Gebäude mit >75% geeigneter Fläche: 205583

Vorschau (Top 10):
      uid_gebaeu  total_roof_area  suitable_roof_area  suitable_share_pct
DEBE00YY10z0001A           252.06              252.06               100.0
DEBE00YY115000B7           156.83              156.83               100.0
DEBE00YY1170005P          2226.53             2226.53               100.0
DEBE00YY1180000H           182.03              182.03               100.0
DEBE00YY11B0005L           184.34              184.34               100.0
DEBE00YY11B0005z           121.42              121.42               100.0
DEBE00YY11B0007U           170.23              170.23               100.0
DEBE00YY11C0004e           134.09              134.09               100.0
DEBE00YY11C0004f            59.15             

In [ ]:
# ============================================================
# EXPORT AS GEODATAFRAME FOR QGIS
# ============================================================

print("\n" + "="*70)
print("Convert to GeoDataFrame and export for QGIS...")
print("="*70)

# Merge with geometries from dachneigung_layer
# Combine ALL geometries per uid_gebaeu (with error handling)
print("Merge geometries (unary_union)...")

def safe_union(geom_series):
    """Merge geometries, fix invalid ones first"""
    try:
        # Validate geometries
        valid_geoms = geom_series[geom_series.is_valid]
        if len(valid_geoms) == 0:
            return None
        # Merge into one polygon
        return valid_geoms.union_all()
    except Exception as e:
        print(f"  ⚠ Error during union: {e}")
        return None

geometry_per_building = dachneigung_layer.groupby('uid_gebaeu').agg({
    'geometry': safe_union  # Merge all segments into one polygon
}).reset_index()

print(f"✓ Geometries merged: {len(geometry_per_building)} buildings")

# Merge building_results_v2 with geometries
building_results_v2_geo = building_results_v2.merge(
    geometry_per_building,
    left_on='uid_gebaeu',
    right_on='uid_gebaeu',
    how='left'
)

# Convert to GeoDataFrame
building_results_v2_geo = gpd.GeoDataFrame(
    building_results_v2_geo,
    geometry='geometry',
    crs=dachneigung_layer.crs
)

print(f"✓ {len(building_results_v2_geo)} features converted to GeoDataFrame")
print(f"  CRS: {building_results_v2_geo.crs}")
print(f"  Columns: {list(building_results_v2_geo.columns)}")

# Save as shapefile
output_shapefile = Path(data_base_path) / "gebaeude_gruendach_potential_v2.shp"
building_results_v2_geo.to_file(output_shapefile)
print(f"\n✓ Shapefile saved: {output_shapefile.name}")

# Save as GeoPackage (better alternative)
output_gpkg = Path(data_base_path) / "gebaeude_gruendach_potential_v2.gpkg"
building_results_v2_geo.to_file(output_gpkg, driver='GPKG')
print(f"✓ GeoPackage saved: {output_gpkg.name}")

print(f"\n✓ Both files are ready for QGIS import!")


Konvertiere zu GeoDataFrame und exportiere für QGIS...
Fasse Geometrien zusammen (unary_union)...
✓ Geometrien zusammengefügt: 529937 Gebäude
✓ 529937 Features als GeoDataFrame konvertiert
  CRS: EPSG:25833
  Spalten: ['uid_gebaeu', 'total_roof_area', 'avg_slope', 'dominant_orientation', 'suitable_roof_area', 'suitable_share_pct', 'geometry']


C:\Users\User\AppData\Local\Temp\ipykernel_21168\1423280320.py:53: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  building_results_v2_geo.to_file(output_shapefile)
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'total_roof_area' to 'total_roof'
  ogr_write(
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'dominant_orientation' to 'dominant_o'
  ogr_write(
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'suitable_roof_area' to 'suitable_r'
  ogr_write(
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roo


✓ Shapefile gespeichert: gebaeude_gruendach_potential_v2.shp
✓ GeoPackage gespeichert: gebaeude_gruendach_potential_v2.gpkg

✓ Beide Dateien sind bereit für QGIS-Import!


In [ ]:
# Load the files back in for further processing
building_results_v2_geo_path = Path(data_base_path) / "gebaeude_gruendach_potential_v2.shp"
building_results_v2_geo = gpd.read_file(building_results_v2_geo_path)

## Multi-Layer Spatial Analysis

Perform spatial joins to merge information from all three layers:
1. Buildings with green roof characteristics
2. Buildings with solar potential
3. Buildings with roof slope analysis

This creates a comprehensive dataset combining green roof potential, roof slopes, and solar generation possibilities.

In [ ]:
# ============================================================
# MERGE THE THREE LAYERS
# ============================================================

print("="*70)
print("Merge the three layers...")
print("="*70)

# Create points from roof segments
green_roofs_points = green_roofs.copy()
green_roofs_points['geometry'] = green_roofs_points['geometry'].representative_point()

print(f"columns in green_roofs_points: {green_roofs_points.columns}")

# STEP 1: Spatial join with green_roofs (intersects)
print("\nStep 1: Merge building_results_v2_geo with green_roofs...")
merged_layer = gpd.sjoin(
    building_results_v2_geo,
    green_roofs_points,
    how='left',  # Left join: keep all building_results, even if no match
    predicate='contains'
)
print(f"✓ {len(merged_layer)} features after green_roofs merge")
print(f"  Columns: {list(merged_layer.columns)}")

# Remove duplicate columns (index_right is from sjoin)
if 'index_right' in merged_layer.columns:
    merged_layer = merged_layer.drop(columns=['index_right'])

# Prepare points for next join (if needed)
solar_potential_layer_points = solar_potential_layer.copy()
solar_potential_layer_points['geometry'] = solar_potential_layer_points['geometry'].representative_point()
print(solar_potential_layer_points.columns)

# STEP 2: Spatial join with solar_potential_layer
print("\nStep 2: Merge result with solar_potential_layer...")
merged_layer = gpd.sjoin(
    merged_layer,
    solar_potential_layer_points,
    how='left',  # Left join: keep all, even if no match
    predicate='contains'
)
print(f"✓ {len(merged_layer)} features after solar_potential merge")

# Remove duplicate columns
if 'index_right' in merged_layer.columns:
    merged_layer = merged_layer.drop(columns=['index_right'])

# Clean up columns (remove geometry columns if multiple)
print(f"\nTotal columns: {len(merged_layer.columns)}")
print(f"Columns: {list(merged_layer.columns)}")

# STEP 3: Export as shapefile
print("\nStep 3: Export as shapefile...")
output_merged = Path(data_base_path) / "merged_gebaeude_gruendach_solar.shp"
merged_layer.to_file(output_merged)
print(f"✓ Shapefile saved: {output_merged.name}")
print(f"  Number of features: {len(merged_layer)}")
print(f"  Number of columns: {len(merged_layer.columns)}")

# Export also as GeoPackage
output_merged_gpkg = Path(data_base_path) / "merged_gebaeude_gruendach_solar.gpkg"
merged_layer.to_file(output_merged_gpkg, driver='GPKG')
print(f"✓ GeoPackage saved: {output_merged_gpkg.name}")

print(f"\n✓ Merging completed! Ready for QGIS review.")

Verschneide die drei Layer...
spalten in green_roofs_points: Index(['geometry', 'gruendach'], dtype='str')

Schritt 1: Verschneide building_results_v2_geo mit green_roofs...
✓ 530485 Features nach green_roofs-Merge
  Spalten: ['uid_gebaeu', 'total_roof', 'avg_slope', 'dominant_o', 'suitable_r', 'suitable_s', 'geometry', 'index_right', 'gruendach']
Index(['geometry', 'anzahl_obe', 'ist_hochha', 'ist_denkma', 'verschattu',
       'verschat_1', 'verschat_2', 'gebaeudefu', 'bauweise_s', 'id'],
      dtype='str')

Schritt 2: Verschneide Ergebnis mit solar_potential_layer...
✓ 530758 Features nach solar_potential-Merge

Gesamt-Spalten: 17
Spalten: ['uid_gebaeu', 'total_roof', 'avg_slope', 'dominant_o', 'suitable_r', 'suitable_s', 'geometry', 'gruendach', 'anzahl_obe', 'ist_hochha', 'ist_denkma', 'verschattu', 'verschat_1', 'verschat_2', 'gebaeudefu', 'bauweise_s', 'id']

Schritt 3: Exportiere als Shapefile...


PermissionError: [WinError 32] Der Prozess kann nicht auf die Datei zugreifen, da sie von einem anderen Prozess verwendet wird: 'C:\\Users\\User\\Documents\\Uni\\Master_Bauing\\WiSe25_26\\AI_in_Human_Water\\berlin-green-roofs\\data\\merged_gebaeude_gruendach_solar.shp'